<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03c_visualizations_net_change.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing

In [69]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px


from google.colab import drive
drive.mount('/content/drive')

!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
import functions as fx

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-04-12 20:42:20--  https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2015 (2.0K) [text/plain]
Saving to: ‘functions.py.3’

functions.py.3      100%[===================>]   1.97K  --.-KB/s    in 0s      

2026-04-12 20:42:21 (31.7 MB/s) - ‘functions.py.3’ saved [2015/2015]



##Reading in file

In [70]:
#gdf of opening dates after 2016 concatenated with gdf of closing dates after 2016
gdf_open_close = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/open_close_after_2016.geojson")

#all businesses with no end date or end dates after 2016
gdf_biz = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/biz_all_startdate.geojson")

#immediately clipping to exclude openings in 2026
gdf_biz = gdf_biz[(gdf_biz.year_open <=2025)]

##Reading in census tracts and block groups

In [72]:
sf_tracts = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.geojson')

sf_block_grp = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_block_grp.geojson')

##Joining census tract geometry and grouping by the tract and the year
- pivoting table to create columns of num openings and num closings

In [73]:
open_close_tracts_gdf = fx.group_points_by_poly_year(gdf_open_close, sf_tracts)

##Calculating new metrics


In [74]:
## net change (openings-closings)

open_close_tracts_gdf['net_change'] = open_close_tracts_gdf['opened'] - open_close_tracts_gdf['closed']

# get 2016 baseline of net change for each geometry
baseline = open_close_tracts_gdf[open_close_tracts_gdf['year'] == 2016][['GEOID', 'net_change']].rename(columns={'net_change': 'baseline_2016'})

# merge basline into gdf of openings closings and tracts
open_close_tracts_gdf = open_close_tracts_gdf.merge(baseline, on='GEOID')

#calculating percent chg in growth from baseline of 2016
open_close_tracts_gdf['growth_pct_over_2016'] = (open_close_tracts_gdf['net_change'] / open_close_tracts_gdf['baseline_2016']) * 100

In [76]:
##getting total number of businesses active in each year ()

#first filling year_closed with 2025 in order to include active businesses in the range
gdf_biz['year_closed'] = gdf_biz['year_closed'].fillna(2025).astype(int)
gdf_biz['year_open'] = gdf_biz['year_open'].astype(int)

#creating an active_years list for each business, which includes an integer of each year it was active at all
gdf_biz['active_years'] = gdf_biz.apply(
    lambda row: list(range(row['year_open'], row['year_closed'] + 1)), axis=1
)

# explode takes a column of lists and creates a row for each item in the column, but still indexed by the same other info/columns
##so here, it's creating a row for each active year of the business
biz_exploded = gdf_biz.explode('active_years').rename(columns={'active_years': 'year'})

#joining this exploded gdf with tract/grp GEOID of its location
biz_exploded = gpd.sjoin(biz_exploded, sf_tracts[['GEOID', 'geometry']], how='left', predicate='within')

#grouping by geoid and year and counting the number of businesses in each year
biz_stock = biz_exploded.groupby(['GEOID', 'year']).size().reset_index(name='biz_stock')

#joining that grouped df into open_close_tract_gdf, which is already grouped by geoid and year
##joining on the left, which means biz_stock rows for years not included in open_close will not be carried over
open_close_tracts_gdf = open_close_tracts_gdf.merge(biz_stock, on=['GEOID', 'year'], how='left')

#And now I'm finally calculating net entry rate for each tract/grp and year
open_close_tracts_gdf['net_entry_rate'] = (open_close_tracts_gdf['net_change'] / open_close_tracts_gdf['biz_stock'])*100

#also throwing in gross exit rate, to help show how much turnover there was in relation to the net entry
open_close_tracts_gdf['gross_exit_rate'] = (open_close_tracts_gdf['opened'] / open_close_tracts_gdf['biz_stock'])*100

#Choropleth maps

###getting epc tracts and adding is_epc boolean

In [83]:
epc_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/epc_tracts_sf.geojson")
epc_tracts = epc_tracts.rename(columns={'tract_geoid': 'GEOID'})
open_close_tracts_gdf['is_epc'] = open_close_tracts_gdf['GEOID'].isin(epc_tracts['GEOID'])

###Net entry rate map (net change for year / total businesses active that year)

In [91]:
# import plotly.graph_objects as go

# plot_gdf = open_close_tracts_gdf[open_close_tracts_gdf['year'] >= 2016].copy()


# vabs = open_close_tracts_gdf['net_entry_rate'].abs().quantile(.99)

# fig_over_time = px.choropleth_mapbox(
#     plot_gdf,
#     geojson=plot_gdf.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="net_entry_rate",
#     hover_name="GEOID",
#     hover_data={'is_epc': True, 'opened': True, 'closed': True, 'gross_exit_rate': True, 'biz_stock':True},
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdBu",
#     color_continuous_midpoint=0,
#     range_color=[-vabs, vabs],
#     height=600,
#     width=700
# )

# epc_outline = plot_gdf[plot_gdf['is_epc']]  # fixed: filter from plot_gdf
# fig_over_time.add_trace(go.Choroplethmapbox(
#     geojson=epc_outline.set_index("GEOID").__geo_interface__,
#     locations=epc_outline["GEOID"],
#     z=[1] * len(epc_outline),
#     colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
#     marker_line_color='black',
#     marker_line_width=3,
#     showscale=False,
#     hoverinfo='skip',
#     name='EPC Tracts'
# ))

# fig_over_time.show()


###Map of percent change in businesses compared to 2016 (net change each year / baseline net change in 2016)

In [90]:

# plot_gdf = open_close_tracts_gdf[open_close_tracts_gdf['year'] >= 2017].copy()

# vabs = plot_gdf['growth_pct_over_2016'].abs().quantile(0.95)

# fig_over_time = px.choropleth_mapbox(
#     plot_gdf,
#     geojson=plot_gdf.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="growth_pct_over_2016",
#     hover_name="GEOID",
#     hover_data={'is_epc': True, 'opened': True, 'closed': True, 'total_activity': True},
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdBu",
#     color_continuous_midpoint=0,
#     range_color=[-vabs, vabs],
#     height=600,
#     width=700
# )

# epc_outline = plot_gdf[plot_gdf['is_epc']]
# fig_over_time.add_trace(go.Choroplethmapbox(
#     geojson=epc_outline.set_index("GEOID").__geo_interface__,
#     locations=epc_outline["GEOID"],
#     z=[1] * len(epc_outline),
#     colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
#     marker_line_color='black',
#     marker_line_width=3,
#     showscale=False,
#     hoverinfo='skip',
#     name='EPC Tracts'
# ))

# fig_over_time.show()
